# SASV: AASIST weighted fusion `α·s_asv + (1−α)·s_cm`

Same idea as notebook `07`, but on **ECAPA + AASIST** scores (no GPU rescoring):

- `runs/ecapa_plus_aasist_dev/scores_dev.csv`
- `runs/ecapa_plus_aasist_eval/scores_eval.csv`

**Protocol**

1. Sweep `α` on **dev** (min SASV-EER)
2. Lock α; apply once on **eval**

Also reports raw sum `s_asv + s_cm` (notebooks 09/10 baseline).

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "weighted_fusion_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

from experiment_lib import DEFAULT_SASV, RUNS_DIR, ensure_sasv_on_path
from weighted_fusion_lib import (
    eers_for_alpha,
    load_score_csv,
    save_weighted_run,
    sweep_alpha,
)

ensure_sasv_on_path(DEFAULT_SASV)
from metrics import get_all_EERs

DEV_CSV = RUNS_DIR / "ecapa_plus_aasist_dev" / "scores_dev.csv"
EVAL_CSV = RUNS_DIR / "ecapa_plus_aasist_eval" / "scores_eval.csv"
OUT = RUNS_DIR / "ecapa_plus_aasist_weighted"
OUT.mkdir(parents=True, exist_ok=True)
print("dev:", DEV_CSV.exists(), "eval:", EVAL_CSV.exists())

In [ ]:
N_GRID = 21
ALPHAS = np.linspace(0.0, 1.0, N_GRID)
RUN_EVAL = True

## 1. Load **dev** + raw-sum reference

In [ ]:
s_asv_dev, s_cm_dev, keys_dev = load_score_csv(DEV_CSV)
print(f"dev trials: {len(keys_dev)}")
sasv_sum, sv_sum, spf_sum = get_all_EERs((s_asv_dev + s_cm_dev).tolist(), keys_dev)
print(
    f"dev raw sum  SASV={sasv_sum*100:.4f}%  SV={sv_sum*100:.4f}%  SPF={spf_sum*100:.4f}%"
)

## 2. Sweep α on **dev**

In [ ]:
best_dev, sweep_rows = sweep_alpha(
    s_asv_dev, s_cm_dev, keys_dev, alphas=ALPHAS, sasv_root=DEFAULT_SASV
)
LOCKED_ALPHA = float(best_dev["alpha"])
print("Locked α:", LOCKED_ALPHA)
print(
    f"dev weighted  SASV={best_dev['sasv_eer_percent']:.4f}%  "
    f"SV={best_dev['sv_eer_percent']:.4f}%  SPF={best_dev['spf_eer_percent']:.4f}%"
)
print("\nα sweep (dev):")
for row in sweep_rows:
    mark = " <-- best" if abs(row["alpha"] - LOCKED_ALPHA) < 1e-9 else ""
    print(
        f"  α={row['alpha']:.2f}  SASV={row['sasv_eer_percent']:7.4f}%  "
        f"SV={row['sv_eer_percent']:7.4f}%  SPF={row['spf_eer_percent']:7.4f}%{mark}"
    )

In [ ]:
dev_out = save_weighted_run(
    split="dev",
    alpha=LOCKED_ALPHA,
    s_asv=s_asv_dev,
    s_cm=s_cm_dev,
    keys=keys_dev,
    metrics={**best_dev, "system": "ecapa_plus_aasist_weighted"},
    output_dir=OUT / "dev",
    source_csv=DEV_CSV,
)
(OUT / "alpha_sweep_dev.json").write_text(
    json.dumps({"locked_alpha": LOCKED_ALPHA, "sweep": sweep_rows}, indent=2),
    encoding="utf-8",
)
print("Wrote", dev_out)

## 3. Locked **eval**

In [ ]:
if not RUN_EVAL:
    print("RUN_EVAL=False")
else:
    s_asv_ev, s_cm_ev, keys_ev = load_score_csv(EVAL_CSV)
    print(f"eval trials: {len(keys_ev)} | α={LOCKED_ALPHA}")
    sasv_s, sv_s, spf_s = get_all_EERs((s_asv_ev + s_cm_ev).tolist(), keys_ev)
    print(
        f"eval raw sum  SASV={sasv_s*100:.4f}%  SV={sv_s*100:.4f}%  SPF={spf_s*100:.4f}%"
    )
    eval_metrics = eers_for_alpha(
        s_asv_ev, s_cm_ev, keys_ev, LOCKED_ALPHA, sasv_root=DEFAULT_SASV
    )
    print(
        f"eval weighted α={LOCKED_ALPHA:.2f}  "
        f"SASV={eval_metrics['sasv_eer_percent']:.4f}%  "
        f"SV={eval_metrics['sv_eer_percent']:.4f}%  "
        f"SPF={eval_metrics['spf_eer_percent']:.4f}%"
    )
    eval_out = save_weighted_run(
        split="eval",
        alpha=LOCKED_ALPHA,
        s_asv=s_asv_ev,
        s_cm=s_cm_ev,
        keys=keys_ev,
        metrics={**eval_metrics, "system": "ecapa_plus_aasist_weighted"},
        output_dir=OUT / "eval",
        source_csv=EVAL_CSV,
    )
    locked = {
        "locked_alpha": LOCKED_ALPHA,
        "tuned_on": "dev",
        "objective": "min SASV-EER",
        "dev_metrics": best_dev,
        "eval_metrics": eval_metrics,
        "eval_raw_sum": {
            "sasv_eer_percent": sasv_s * 100,
            "sv_eer_percent": sv_s * 100,
            "spf_eer_percent": spf_s * 100,
        },
    }
    (OUT / "locked_eval.json").write_text(json.dumps(locked, indent=2), encoding="utf-8")
    print("Wrote", eval_out)
    print(json.dumps(locked, indent=2))

## Done

Outputs under `runs/ecapa_plus_aasist_weighted/`.  
Next optional: notebook `12` (AASIST + LFCC CM ensemble).